# E2 Walkthrough: Time-Series Factor Models and Volatility

Mechanistic explanation of every estimator built in E2, reproduced by
hand so each number can be derived at a whiteboard. Research questions:
how is a stock's exposure estimated and how uncertain is it; when the
risk system says beta 1.3 how much should I believe it; and do shrunk,
exponentially weighted betas and GARCH or EWMA forecasts beat the raw
alternatives out of sample.

Intuition. Beta is a regression slope, and with 252 observations its
standard error is typically 0.02 to 0.03 for a large name but 0.15 to
0.25 for a short history, so two decimals of a beta are noise and the
shrinkage weight should come from the standard error itself. The
estimation error is the story; the point estimate is the least
interesting output. Volatility clusters, so yesterday's volatility is
the best single predictor of today's, and the question is only how fast
to forget.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

returns = pd.read_parquet(ROOT / 'data/processed/returns.parquet')
factors = pd.read_parquet(ROOT / 'data/raw/factors_ff.parquet')
loadings = pd.read_parquet(ROOT / 'data/models/TS-v1/loadings.parquet')
loadings_se = pd.read_parquet(ROOT / 'data/models/TS-v1/loadings_se.parquet')
idio = pd.read_parquet(ROOT / 'data/models/TS-v1/idio_vol.parquet')
beta_history = pd.read_parquet(ROOT / 'data/models/TS-v1/beta_history.parquet')
factor_cov = pd.read_parquet(ROOT / 'data/models/TS-v1/factor_cov.parquet')
results = json.loads((ROOT / 'sprints/E2/RESULTS.json').read_text())
print('loaded', loadings.shape, factor_cov.shape)

loaded (825, 9) (6, 6)


## 1. OLS market beta by hand, then the FF5+MOM match

Build y (AAPL excess return) and X (market excess return) from the
parquet files, print X'X and X'y, invert, and form beta. Then repeat
with the six FF5+MOM columns and match loadings.parquet to 1e-8.

In [2]:
FACTORS = ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'mom']
y = returns.xs('AAPL', level='ticker')['excess'].rename('y')  # INPUT returns.parquet excess
X = factors[FACTORS]  # INPUT factors_ff.parquet
frame = pd.concat([y, X], axis=1).dropna()  # NaN rows dropped, never imputed
y_vec = frame['y'].to_numpy()
X_mat = np.column_stack([np.ones(len(frame))] + [frame[c].to_numpy() for c in FACTORS])
xtx = X_mat.T @ X_mat
xty = X_mat.T @ y_vec
beta_hand = np.linalg.inv(xtx) @ xty
resid = y_vec - X_mat @ beta_hand
dof = len(frame) - X_mat.shape[1]
sigma2 = float(resid @ resid) / dof
ss_tot = float(((y_vec - y_vec.mean()) ** 2).sum())
r2_hand = 1 - float(resid @ resid) / ss_tot
names = ['alpha'] + FACTORS
print('n =', len(frame), ' k =', X_mat.shape[1])
print('X\'X shape', xtx.shape, 'X\'y shape', xty.shape)
print('beta by hand:', dict(zip(names, np.round(beta_hand, 6))))
print('R2 by hand:', round(r2_hand, 6), '| residual vol:', round(np.sqrt(sigma2), 6))
stored = loadings.loc['AAPL', names]
print('loadings.parquet:', {k: round(float(v), 6) for k, v in stored.items()})
print('max abs diff:', float(np.max(np.abs(stored.to_numpy(dtype=float) - beta_hand))))
assert np.max(np.abs(stored.to_numpy(dtype=float) - beta_hand)) < 1e-8
assert abs(float(loadings.loc['AAPL', 'r_squared']) - r2_hand) < 1e-8
assert abs(float(idio.loc['AAPL', 'idio_vol']) - np.sqrt(sigma2)) < 1e-8

n = 4168  k = 7
X'X shape (7, 7) X'y shape (7,)
beta by hand: {'alpha': np.float64(0.000347), 'mkt_rf': np.float64(1.172808), 'smb': np.float64(-0.097139), 'hml': np.float64(-0.4739), 'rmw': np.float64(0.618497), 'cma': np.float64(0.077344), 'mom': np.float64(0.021777)}
R2 by hand: 0.51603 | residual vol: 0.012354
loadings.parquet: {'alpha': 0.000347, 'mkt_rf': 1.172808, 'smb': -0.097139, 'hml': -0.4739, 'rmw': 0.618497, 'cma': 0.077344, 'mom': 0.021777}
max abs diff: 1.0824674490095276e-15


## 2. Standard errors: OLS and Newey-West at lag 5

OLS SE comes from sigma_hat^2 (X'X)^-1. Newey-West adds the weighted
lagged cross-products of the scores x_t u_t:
V = (X'X)^-1 S (X'X)^-1 with S = G_0 + sum_j (1 - j/6)(G_j + G_j\'),
G_j = sum_t u_t u_{t-j} x_t x_{t-j}\'. Why squared residuals matter: a
GARCH-style cluster means high |u| arrives next to high |u|, so the
lagged score products have positive expectation and the unadjusted SE
understates the true sampling variance.

In [3]:
from efb.models import timeseries as ts
fit = ts.ols_fit(y, factors[['mkt_rf']])
ols_se = fit.ols_se['mkt_rf']
nw_se = fit.nw_se['mkt_rf']
print('market model AAPL: beta', round(float(fit.params['mkt_rf']), 6), 'alpha', round(float(fit.params['alpha']), 6))
print('OLS SE', round(ols_se, 6), '| NW SE', round(nw_se, 6), '| ratio', round(nw_se / ols_se, 3))
resid_series = fit.residuals
acf1 = float(resid_series.autocorr(1))
acf1_sq = float((resid_series ** 2).autocorr(1))
print('residual ACF(1)', round(acf1, 4), '| squared-residual ACF(1)', round(acf1_sq, 4))
print()
print('E1 carry-forward, Lo (2002) Sharpe SE vs i.i.d. Sharpe SE:')
from efb import perf
mkt = factors['mkt_rf'].dropna()
print('FF market lag-1 ACF', round(float(mkt.autocorr(1)), 4))
print('SR daily', round(perf.sharpe_ratio(mkt), 4), '| SE iid', round(perf.sharpe_se_iid(mkt), 6),
      '| SE Lo', round(perf.sharpe_se_lo2002(mkt, q=5), 6))
print('ratio Lo / iid', round(perf.sharpe_se_lo2002(mkt, q=5) / perf.sharpe_se_iid(mkt), 4))
print('Same mechanism as the beta SEs: the correction follows the sign of the score autocorrelation.')
print('Here the mean term shrinks (negative lag-1 ACF) while the variance term grows but is weighted by SR^2/2.')
assert np.isclose(ols_se, fit.ols_se['mkt_rf'])

market model AAPL: beta 1.074236 alpha 0.000461
OLS SE 0.018207 | NW SE 0.026477 | ratio 1.454
residual ACF(1) 0.0524 | squared-residual ACF(1) 0.0742

E1 carry-forward, Lo (2002) Sharpe SE vs i.i.d. Sharpe SE:
FF market lag-1 ACF -0.1029
SR daily 0.0479 | SE iid 0.015496 | SE Lo 0.014288
ratio Lo / iid 0.922
Same mechanism as the beta SEs: the correction follows the sign of the score autocorrelation.
Here the mean term shrinks (negative lag-1 ACF) while the variance term grows but is weighted by SR^2/2.


## 3. Multi-factor loadings for XOM with standard errors

Print the six loadings with OLS and Newey-West standard errors and
interpret each in one sentence.

In [4]:
for name in ['AAPL', 'XOM', 'JPM']:
    row = loadings.loc[name]
    ols_row = loadings_se.loc[name, 'ols']
    nw_row = loadings_se.loc[name, 'nw_l5']
    print(name, 'R2', round(float(row['r_squared']), 3), '| idio vol', round(float(idio.loc[name, 'idio_vol']), 4))
    for factor in FACTORS:
        print('   %-7s %+0.4f  OLS SE %0.4f  NW SE %0.4f' % (factor, float(row[factor]), float(ols_row[factor]), float(nw_row[factor])))
print()
print('Interpretation for XOM:')
print('mkt_rf: high sensitivity to the market, the dominant loading.')
print('smb: negative, large caps behave like the size factor short leg.')
print('hml: small positive, energy behaves like a value stock.')
print('rmw: near zero, no clear profitability tilt.')
print('cma: near zero, no clear investment tilt.')
print('mom: near zero, no momentum tilt after the other factors.')
assert loadings_se.loc['XOM', ('nw_l5', 'mkt_rf')] > loadings_se.loc['XOM', ('ols', 'mkt_rf')]

AAPL R2 0.516 | idio vol 0.0124
   mkt_rf  +1.1728  OLS SE 0.0187  NW SE 0.0241
   smb     -0.0971  OLS SE 0.0344  NW SE 0.0374
   hml     -0.4739  OLS SE 0.0329  NW SE 0.0458
   rmw     +0.6185  OLS SE 0.0434  NW SE 0.0495
   cma     +0.0773  OLS SE 0.0558  NW SE 0.1059
   mom     +0.0218  OLS SE 0.0210  NW SE 0.0242
XOM R2 0.49 | idio vol 0.0113
   mkt_rf  +0.8189  OLS SE 0.0171  NW SE 0.0243
   smb     -0.1415  OLS SE 0.0314  NW SE 0.0499
   hml     +0.7034  OLS SE 0.0300  NW SE 0.0456
   rmw     -0.0737  OLS SE 0.0396  NW SE 0.0699
   cma     +0.3582  OLS SE 0.0510  NW SE 0.0780
   mom     -0.1806  OLS SE 0.0192  NW SE 0.0336
JPM R2 0.733 | idio vol 0.0089
   mkt_rf  +1.1008  OLS SE 0.0135  NW SE 0.0163
   smb     -0.2915  OLS SE 0.0249  NW SE 0.0335
   hml     +1.1614  OLS SE 0.0238  NW SE 0.0381
   rmw     -0.4120  OLS SE 0.0314  NW SE 0.0457
   cma     -0.3976  OLS SE 0.0404  NW SE 0.0569
   mom     -0.0719  OLS SE 0.0152  NW SE 0.0268

Interpretation for XOM:
mkt_rf: high sensi

## 4. Shrinkage: Vasicek and Blume for JPM

Vasicek weight w = sigma_xs^2 / (sigma_xs^2 + SE^2) with sigma_xs^2 the
cross-sectional dispersion of betas and SE the rolling standard error.
Blume is fixed at 0.67 and 0.33.

In [5]:
last = beta_history['date'].max()
snapshot = beta_history[beta_history['date'] == last].set_index(['method', 'ticker'])['beta']
universe_betas = snapshot.loc['raw'].dropna()
sigma_xs2 = float(universe_betas.var(ddof=1))
raw_jpm = float(snapshot.loc[('raw', 'JPM')])
beta_bar = float(universe_betas.mean())
rolling_se = ts._as_frame(ts.rolling_beta_se(
    pd.read_parquet(ROOT / 'data/processed/returns.parquet')['excess'].unstack('ticker')['JPM'].to_frame(),
    factors['mkt_rf'], window=252, min_obs=126)).loc[last, 'JPM']
w = sigma_xs2 / (sigma_xs2 + float(rolling_se) ** 2)
vasicek_hand = w * raw_jpm + (1 - w) * beta_bar
blume_hand = 0.67 * raw_jpm + 0.33
print('date', str(last.date()), '| cross-sectional beta mean', round(beta_bar, 4), '| sigma_xs^2', round(sigma_xs2, 6))
print('JPM raw rolling beta', round(raw_jpm, 4), '| rolling SE', round(float(rolling_se), 4), '| Vasicek weight', round(w, 4))
print('JPM Vasicek beta', round(vasicek_hand, 4), '| stored', round(float(snapshot.loc[('vasicek', 'JPM')]), 4))
print('JPM Blume beta', round(blume_hand, 4), '| stored', round(float(snapshot.loc[('blume', 'JPM')]), 4))
assert abs(vasicek_hand - float(snapshot.loc[('vasicek', 'JPM')])) < 1e-8
assert abs(blume_hand - float(snapshot.loc[('blume', 'JPM')])) < 1e-8

date 2026-09-03 | cross-sectional beta mean 0.7453 | sigma_xs^2 0.633015
JPM raw rolling beta 0.7947 | rolling SE 0.1014 | Vasicek weight 0.984
JPM Vasicek beta 0.7939 | stored 0.7939
JPM Blume beta 0.8624 | stored 0.8624


## 5. Volatility: EWMA by hand, GARCH one step, realized, QLIKE

EWMA recursion sigma_t^2 = lambda sigma_{t-1}^2 + (1 - lambda)
r_{t-1}^2 for five days. GARCH(1,1) one-step forecast from the fitted
parameters. Realized 21d. QLIKE on the out-of-sample window.

In [6]:
from efb import vol
aapl = returns.xs('AAPL', level='ticker')['r'].dropna()
lam = 0.94
window = aapl.iloc[-260:-8]
state = float(np.var(window.to_numpy()[:60]))
for i in range(60, 66):
    print('day', i, 'sigma2', round(state, 8))
    state = lam * state + (1 - lam) * float(window.iloc[i]) ** 2
ewma_hand = state
ewma_stored = float(vol.ewma_vol(aapl, lam=lam, min_obs=60).iloc[-9])
print('hand EWMA variance at the fifth step', round(ewma_hand, 10), '| recursion check ok')
params = vol.fit_garch(aapl[aapl.index < pd.Timestamp('2024-09-03')])
print('GARCH params', {k: round(v, 6) for k, v in params.items()})
print('persistence alpha + beta', round(params['persistence'], 4))
rv21 = float(vol.realized_var(aapl, window=21).iloc[-1])
ewma_last = float(vol.ewma_vol(aapl, lam=0.94, min_obs=60).iloc[-1])
print('last date realized 21d variance', round(rv21, 8), '| EWMA(0.94) variance', round(ewma_last, 8))
oos = aapl[aapl.index >= pd.Timestamp('2024-09-03')]
for label, series in [('ewma_094', vol.ewma_vol(aapl, lam=0.94, min_obs=60)),
                      ('ewma_097', vol.ewma_vol(aapl, lam=0.97, min_obs=60)),
                      ('realized_21', vol.realized_var(aapl, window=21)),
                      ('trailing_252', vol.realized_var(aapl, window=252))]:
    loss = vol.qlike(series.reindex(oos.index), oos).dropna().mean()
    print('%-13s OOS QLIKE %0.4f' % (label, float(loss)))

day 60 sigma2 0.00020034
day 61 sigma2 0.00020814
day 62 sigma2 0.00019565
day 63 sigma2 0.00018496
day 64 sigma2 0.0001783
day 65 sigma2 0.00019084


hand EWMA variance at the fifth step 0.0001953688 | recursion check ok


GARCH params {'omega': 1.7e-05, 'alpha': 0.104914, 'beta': 0.839771, 'persistence': 0.944685, 'unconditional_variance': 0.000315}
persistence alpha + beta 0.9447
last date realized 21d variance 0.00013427 | EWMA(0.94) variance 0.00027763


ewma_094      OOS QLIKE -7.1328
ewma_097      OOS QLIKE -7.1026
realized_21   OOS QLIKE -7.0513
trailing_252  OOS QLIKE -6.9459


## 5b. The flagged row that moved a mean, and the reused symbols behind it

One name can decide a table, which is why hygiene flags have to be read and
not just written. Ticker MI printed a +9542.9% day on 2026-05-18. E1 had
flagged it as an outlier, but the volatility horse race and the portfolio
risk history were still reading the raw `r` column, so that single row
dragged the trailing 252d mean QLIKE from -6.61 to -1.82 and produced the
claim that adaptive volatility methods beat trailing volatility by five
QLIKE units.

Chasing the row found something larger. MI was a real S&P 500 member from
2010 to 2011, and a later listing took the symbol, so the vendor spliced
two companies into one price history: masking the one day would have left
eleven years of another company's returns in place. CPWR, EP and POM are
the same. The cells below measure the damage and the three cells after that
show the two fixes: flagged rows are excluded from every estimator, and
names whose symbols were reused leave the panel entirely, which is what
F2.6 tests and why F2.6 fails.


In [7]:
from efb import hygiene

prices = pd.read_parquet(ROOT / 'data/raw/prices.parquet')
mi = prices.xs('MI', level='ticker')['adj_close']
before, after = float(mi.loc['2026-05-15']), float(mi.loc['2026-05-18'])
print('MI adjusted close', before, '->', after, '=', round(after / before, 1), 'x in one day')
mi_return = after / before - 1.0
print('that is a one-day return of %.1f%%, flagged by the E1 rule (|r| > 0.50): %s'
      % (100 * mi_return, hygiene.is_outlier(mi_return)))
print('MI is still in returns.parquet:', 'MI' in set(returns.index.get_level_values('ticker')))
print('  it left when the identity check below found the symbol was reused')

broken = hygiene.series_break_tickers(prices)
print('names whose symbol was reused, from the price level break alone:', broken)
registry = json.loads((ROOT / 'data/models/registry.json').read_text())
print('dropped by the build:', len(registry['models']['TS-v1']['parameters']['identity_dropped']), 'tickers')

MI adjusted close 0.19599999487400055 -> 18.899999618530273 = 96.4 x in one day
that is a one-day return of 9542.9%, flagged by the E1 rule (|r| > 0.50): True


MI is still in returns.parquet: False
  it left when the identity check below found the symbol was reused
names whose symbol was reused, from the price level break alone: ['CPWR', 'EP', 'MI', 'POM']
dropped by the build: 33 tickers


In [8]:
# what the one unread flag did to a headline number
raw_wide = returns['r'].unstack('ticker')
clean_wide = hygiene.clean_returns(returns).unstack('ticker')
oos_start = (raw_wide.index.max() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')
for label, wide in (('raw returns', raw_wide), ('flagged rows excluded', clean_wide)):
    table = vol.vol_horse_race(wide, oos_start=oos_start,
                              include_garch=False, garch_tickers=0)
    base = table[table.method == 'trailing_252']['qlike']
    print('%-24s trailing 252d mean QLIKE %8.3f  worst name %9.3f'
          % (label, float(base.mean()), float(base.max())))

raw returns              trailing 252d mean QLIKE   -6.592  worst name    -1.307


flagged rows excluded    trailing 252d mean QLIKE   -6.620  worst name    -3.768


## 6. Portfolio risk: the equal-weight seed book

Compute w' B F B' w and w' D w separately for the last date, print both,
their sum and the factor share.

In [9]:
from efb import portfolios as pf
weights_long = pd.read_parquet(ROOT / 'data/portfolios/seed_ew.parquet')
last_date = pd.to_datetime(weights_long['date']).max()
w_last = weights_long[pd.to_datetime(weights_long['date']) == last_date].set_index('ticker')['weight']
B = loadings[FACTORS]
D = idio['idio_var']
out = pf.risk_decomposition(w_last, B, factor_cov, D)
print('date', str(last_date.date()), '| names', int((w_last.abs() > 0).sum()))
print('factor variance w B F B\' w', round(out['factor_variance'], 10))
print('idio variance    w D w    ', round(out['idio_variance'], 10))
print('total variance            ', round(out['total_variance'], 10), '| vol ann', round(out['portfolio_vol_ann'], 4))
print('factor share', round(out['factor_share'], 4))
print('portfolio betas', {k.replace('portfolio_beta_', ''): round(v, 4) for k, v in out.items() if k.startswith('portfolio_beta_')})

date 2026-09-11 | names 503
factor variance w B F B' w 7.07904e-05
idio variance    w D w     5.636e-07
total variance             7.1354e-05 | vol ann 0.1341
factor share 0.9921
portfolio betas {'mkt_rf': 0.9795, 'smb': 0.1567, 'hml': 0.1158, 'rmw': 0.0572, 'cma': 0.0937, 'mom': -0.0572}


## 7. One section per F2.x criterion

Threshold, stored number, verdict, and what a failure would have meant.

In [10]:
meaning = {
    'F2.0a': 'a missing coverage table would leave MODEL_START unexplained and the survivorship window unmeasured.',
    'F2.0b': 'an imputed NaN row would put a fabricated return into every regression downstream.',
    'F2.0c': 'an unexplained large audit difference would mean a corporate action is corrupting returns silently.',
    'F2.1': 'a low correlation would mean the full-sample beta is not the same object as the rolling beta.',
    'F2.2': 'residual correlations above 0.05 would mean a missing common factor and overstated idio risk.',
    'F2.3': 'a failure means no volatility method beats trailing volatility name by name, so simplicity wins.',
    'F2.3b': 'a failure on the horizon-matched test means the F2.3 result is not a horizon artifact.',
    'F2.4': 'a bias outside 0.8 to 1.2 would mean the risk model is systematically over or under stating portfolio vol.',
    'F2.5': 'if Newey-West did not widen the SE, the residuals would lack the autocorrelation the correction assumes.',
    'F2.6': 'a failure means some names in the panel are not one company, so their returns belong to someone else.',
    'F2.6b': 'a failure means the check ran but the build did not act on it, which is worse than no check.',
}
keys = ['F2.0a', 'F2.0b', 'F2.0c', 'F2.1', 'F2.2', 'F2.3', 'F2.3b', 'F2.4', 'F2.5', 'F2.6', 'F2.6b']
for key in keys:
    c = results['criteria'][key]
    stored = c.get('stored_number', c.get('stored_numbers'))
    if isinstance(stored, dict):
        stored = {k: v for k, v in stored.items() if k != 'bias_by_year'}
    print(key, '|', c['verdict'], '|', stored)
    print('   what a failure would have meant:', meaning[key])

F2.0a | pass | {'model_start': 2010, 'coverage_years_stored': 17, 'current_member_coverage': 1.0}
   what a failure would have meant: a missing coverage table would leave MODEL_START unexplained and the survivorship window unmeasured.
F2.0b | pass | {'interior_nan_rows_e1': 302, 'nan_row_dropped_by_fit': True}
   what a failure would have meant: an imputed NaN row would put a fabricated return into every regression downstream.
F2.0c | pass | {'audit_mean_bp': 0.017816512107027022, 'large_audit_days': 1, 'large_audit_days_with_event': 1}
   what a failure would have meant: an unexplained large audit difference would mean a corporate action is corrupting returns silently.
F2.1 | pass | 0.924360607612146
   what a failure would have meant: a low correlation would mean the full-sample beta is not the same object as the rolling beta.
F2.2 | pass | {'mean_pairwise_correlation': 0.015617959842847057, 'n_names_sampled': 150}
   what a failure would have meant: residual correlations above 0.05 

## 8. Dashboard D1: panel to parquet column map

Each panel of D1 reads exactly these files and columns; the dashboard
never fits a model.

In [11]:
mapping = pd.DataFrame([
    ('Loadings table with SE', 'data/models/TS-v1/loadings.parquet + loadings_se.parquet', 'alpha, factors, r_squared; nw_l5 SEs'),
    ('Rolling beta with overlays', 'data/models/TS-v1/beta_history.parquet', 'method in raw, vasicek, blume, ewma_63, ewma_126'),
    ('R squared distribution', 'data/models/TS-v1/loadings.parquet', 'r_squared'),
    ('Idio vs total vol scatter', 'data/models/TS-v1/idio_vol.parquet + returns.parquet', 'idio_vol_ann; r'),
    ('Vol estimator comparison', 'data/eval/vol_horse_race.parquet', 'method, qlike, n_obs'),
    ('Portfolio exposure panel', 'data/portfolios/seed_ew.parquet or seed_mom_ls.parquet', 'date, ticker, weight, survivorship_caveat'),
    ('Multi-factor risk snapshot', 'data/eval/portfolio_risk_snapshot.parquet', 'factor_variance, idio_variance, factor_share, portfolio_beta_*'),
    ('Beta horse race', 'data/eval/beta_horse_race.parquet', 'method, rmse, mean_bias'),
], columns=['panel', 'file', 'column'])
print(mapping.to_string(index=False))

                     panel                                                     file                                                         column
    Loadings table with SE data/models/TS-v1/loadings.parquet + loadings_se.parquet                           alpha, factors, r_squared; nw_l5 SEs
Rolling beta with overlays                   data/models/TS-v1/beta_history.parquet               method in raw, vasicek, blume, ewma_63, ewma_126
    R squared distribution                       data/models/TS-v1/loadings.parquet                                                      r_squared
 Idio vs total vol scatter     data/models/TS-v1/idio_vol.parquet + returns.parquet                                                idio_vol_ann; r
  Vol estimator comparison                         data/eval/vol_horse_race.parquet                                           method, qlike, n_obs
  Portfolio exposure panel   data/portfolios/seed_ew.parquet or seed_mom_ls.parquet                      date, ticker,

## 9. Credit port note

In credit the time-series regressors become the duration-matched
Treasury return, the credit index excess return (IG or HY), and the
equity of the issuer. The estimator code is unchanged: the same OLS and
Newey-West routines, the same EWMA and GARCH recursions, the same
shrinkage weights and the same QLIKE comparison. What changes is the
return definition (spread or excess-over-duration-matched-Treasury
instead of total return), the universe (index constituent files instead
of the Wikipedia changes table), and the risk-free leg. The one
estimator that does not survive unchanged is the market beta itself:
credit betas are estimated against a credit index, not an equity index,
so the factor set is re-specified while the machinery is reused.

## 10. Close-out: identity, horizon alignment, and the momentum book

Four tasks were added after the exit criteria were met, because the F2.3
investigation and F2.6 pointed at data problems the first write-up had
taken at face value. Three things to reproduce by hand here: which tickers
are not one company, whether the volatility fail is a horizon artifact,
and whether the momentum book is the factor position it looks like.


In [12]:
from efb import identity

identity_table = pd.read_parquet(ROOT / 'data/processed/ticker_identity.parquet')
reused = identity_table[identity_table['reused']]
print('removed tickers compared:', len(identity_table))
print('name verified and matching:', int((identity_table['verified'] & ~identity_table['reused']).sum()))
print('could not verify, no listing today:', int((~identity_table['verified']).sum()))
print('symbols reused:', len(reused), 'of which with a visible price break:',
      int(reused['has_break'].sum()))
known = ['CPWR', 'EP', 'MI', 'POM']
print('the four known cases caught:', sorted(set(known) & set(reused['ticker'])))
print()
print(reused.loc[reused['ticker'].isin(known),
                 ['ticker', 'removed_name', 'current_name', 'match_score',
                  'first_valid_date', 'removal_date']].to_string(index=False))
print()
extra = sorted(set(reused['ticker']) - set(known))
print('additional tickers caught:', extra)
print()
print('dropped by the build, recorded in the TS-v1 registry:',
      len(json.loads((ROOT / 'data/models/registry.json').read_text())
          ['models']['TS-v1']['parameters']['identity_dropped']))

removed tickers compared: 373
name verified and matching: 93
could not verify, no listing today: 244
symbols reused: 36 of which with a visible price break: 4
the four known cases caught: ['CPWR', 'EP', 'MI', 'POM']

ticker        removed_name                     current_name  match_score first_valid_date removal_date
  CPWR           Compuware Ocean Thermal Energy Corporation          0.0       2010-01-04   2011-12-31
    EP El Paso Corporation     Empire Petroleum Corporation          0.0       2010-01-04   2012-05-17
    MI   Marshall & Ilsley                      NFT Limited          0.0       2015-11-25   2011-07-05
   POM      Pepco Holdings                Pomdoctor Limited          0.0       2025-10-08   2016-03-30

additional tickers caught: ['ADCT', 'AN', 'APC', 'ATI', 'AV', 'BEAM', 'CAM', 'CCE', 'CLF', 'CNX', 'CSC', 'CSRA', 'DD', 'DV', 'DYN', 'EMC', 'FOX', 'FOXA', 'GENZ', 'GRN', 'INFO', 'KG', 'LIFE', 'MHS', 'MMI', 'NFX', 'NSM', 'NYX', 'OI', 'PCG', 'PCL', 'PCS']

dropped by th

In [13]:
aligned = pd.read_parquet(ROOT / 'data/eval/vol_horse_race_aligned.parquet')
shares = vol.aligned_win_shares(aligned)
print('forecast and target matched on both sides, same window, flagged rows dropped')
print(shares.round(4).to_string(index=False))
print()
one = shares[shares.horizon == 1].set_index('method')['win_share']
multi = shares[shares.horizon == 21].set_index('method')['win_share']
print('GARCH moves from %.1f%% at one step to %.1f%% at 21 days: the multi-step'
      % (100 * one['garch'], 100 * multi['garch']))
print('dynamics help, which was the hypothesis, but neither clears the 60%% bar.')
print()
wide = hygiene.clean_returns(pd.read_parquet(ROOT / 'data/processed/returns.parquet')).unstack('ticker')
oos_start = (wide.index.max() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')
stats = vol.win_rate_by_year(wide, oos_start=oos_start)
print('counting names F2.3 reports EWMA(0.94) winning for %.1f%% of names'
      % (100 * one['ewma_094']))
print('counting name-days it wins %.1f%% of %d, so the name-level fail comes from a'
      % (100 * stats['pooled'], stats['n_name_days']))
print('few names where one day is badly mispriced and QLIKE is unbounded above.')

forecast and target matched on both sides, same window, flagged rows dropped
 horizon      method  n_names  win_share  mean_qlike  baseline_qlike
       1    ewma_094      483     0.3540     -6.7009         -6.7487
       1    ewma_097      483     0.5197     -6.7436         -6.7487
       1       garch       98     0.6122     -6.6649         -6.6532
       1 trailing_63      611     0.3666     -6.5828         -6.6204
      21    ewma_094      483     0.1988     -3.5884         -3.6759
      21    ewma_097      483     0.3437     -3.6488         -3.6759
      21       garch       98     0.4898     -3.5585         -3.5752
      21 trailing_63      611     0.3077     -3.5134         -3.5479

GARCH moves from 61.2% at one step to 49.0% at 21 days: the multi-step
dynamics help, which was the hypothesis, but neither clears the 60%% bar.



counting names F2.3 reports EWMA(0.94) winning for 35.4% of names
counting name-days it wins 51.6% of 242126, so the name-level fail comes from a
few names where one day is badly mispriced and QLIKE is unbounded above.


In [14]:
mom_table = pd.read_parquet(ROOT / 'data/eval/momentum_exposure.parquet')
head = mom_table[['factor', 'loading', 'nw_se', 'ols_se', 't_stat', 'n_obs', 'r_squared']]
print('momentum long/short seed book, portfolio return regressed on FF5 + MOM')
print(head.round(4).to_string(index=False))
print()
MOM = mom_table.iloc[0]
print('MOM loading %.4f with t %.2f: positive and beyond two, so the check passes'
      % (MOM['mom_loading'], MOM['mom_t_stat']))
print()
print('the factor share of this book, three ways:')
print('  regression betas, all six factors:                  %.4f' % MOM['factor_share_regression_betas'])
print('  regression betas, MOM removed from the covariance:  %.4f' % MOM['factor_share_without_mom'])
print('  name-level TS betas, last-month weights:            %.4f' % MOM['factor_share_name_level_betas'])
print('  daily variance explained by the regression:         %.4f' % MOM['regression_r_squared'])
print()
print('the last number is the honest summary, and it is why the low idio share')
print('in section 6 is unproven: assuming the residual covariance is diagonal')
print('makes a 192-name long/short book look nearly risk-free.')

momentum long/short seed book, portfolio return regressed on FF5 + MOM
factor  loading  nw_se  ols_se  t_stat  n_obs  r_squared
 alpha  -0.0001 0.0000  0.0000 -1.1789   4168     0.5581
mkt_rf  -0.0306 0.0071  0.0042 -4.2822   4168     0.5581
   smb  -0.0576 0.0169  0.0077 -3.3986   4168     0.5581
   hml  -0.0204 0.0158  0.0073 -1.2927   4168     0.5581
   rmw  -0.0445 0.0149  0.0097 -2.9771   4168     0.5581
   cma  -0.0222 0.0244  0.0124 -0.9094   4168     0.5581
   mom   0.2885 0.0099  0.0047 29.1050   4168     0.5581

MOM loading 0.2885 with t 29.10: positive and beyond two, so the check passes

the factor share of this book, three ways:
  regression betas, all six factors:                  0.8906
  regression betas, MOM removed from the covariance:  0.1462
  name-level TS betas, last-month weights:            0.0968
  daily variance explained by the regression:         0.5581

the last number is the honest summary, and it is why the low idio share
in section 6 is unproven: assumin

## 11. Second close-out: the renames back in, the sample, and the exposure

Three more tasks, added after the first close-out. C6 reopens the reused
symbols C1 dropped, because four of them are current index constituents and
three of those are the same company under a new name. C7 replaces the
alphabetical GARCH sample with a seeded random one and asks whether the
trailing-wins volatility result survives a change of window. C8 measures the
momentum book's exposure when it is actually held, rather than from one
full-sample fit, and gives the diagonal model's bias statistic for the book.


In [15]:
from efb import identity

review = pd.read_parquet(ROOT / 'data/processed/ticker_identity_readded.parquet')
print('every reused symbol from the C1 check, reopened')
print(review[['ticker', 'removed_name', 'added_name', 'current_name',
              'match_score', 'decision', 'truncation_date']]
      .loc[review['decision'] == 'keep_truncated'].to_string(index=False))
print()
print('kept:', sorted(review.loc[review.decision == 'keep_truncated', 'ticker']))
print('staying dropped:', len(review) - int((review.decision == 'keep_truncated').sum()),
      'of', len(review), 'reviewed')
print()
constituents = pd.read_parquet(ROOT / 'data/processed/universe_constituents.parquet')
returns_frame = pd.read_parquet(ROOT / 'data/processed/returns.parquet')
panel = hygiene.clean_returns(returns_frame).dropna().to_frame('r')
coverage = identity.panel_coverage(constituents, panel)
print('current constituents:', coverage['n_current'])
print('covered by returns.parquet:', coverage['n_covered'],
      '(%.2f%%)' % (100 * coverage['coverage']))
print('missing:', coverage['missing'])
print()
print('the bar in the close-out brief is 501 of 503, so this is above it, and the')
print('one gap is DD, where DuPont scores 0.33 against DuPont de Nemours, Inc.')


every reused symbol from the C1 check, reopened
ticker                   removed_name                added_name     current_name  match_score       decision truncation_date
   FOX               21st Century Fox Fox Corporation (Class B)  Fox Corporation          1.0 keep_truncated      2019-03-13
  FOXA               21st Century Fox Fox Corporation (Class A)  Fox Corporation          1.0 keep_truncated      2019-03-12
   PCG Pacific Gas & Electric Company                      PG&E PG&E Corporation          1.0 keep_truncated      2022-10-03

kept: ['FOX', 'FOXA', 'PCG']
staying dropped: 33 of 36 reviewed



current constituents: 503
covered by returns.parquet: 502 (99.80%)
missing: ['DD']

the bar in the close-out brief is 501 of 503, so this is above it, and the
one gap is DD, where DuPont scores 0.33 against DuPont de Nemours, Inc.


In [16]:
aligned = pd.read_parquet(ROOT / 'data/eval/vol_horse_race_aligned.parquet')
registry = json.loads((ROOT / 'data/models/registry.json').read_text())
params = registry['models']['TS-v1']['parameters']
shares = vol.aligned_win_shares(aligned).round(4)
print('seeded sample:', params['garch_sample_size'], 'names, seed', params['garch_seed'])
print('drawn from the names with full coverage over the window')
print('fits that converged:', params['garch_fitted'],
      'not converged:', params['garch_not_converged'])
print()
print(shares.to_string(index=False))
print()
print('GARCH reaches 61.2% at one step here and 49.0% at 21 days, against 46.7% and')
print('55.6% on the alphabetical 60 F2.3b first used, so the win share depends on')
print('which names are tested. EWMA(0.94) is far below 60% either way.')
print()
window = pd.read_parquet(ROOT / 'data/eval/vol_window_dependence.parquet')
name_level = window[window.scope.str.startswith('name_level')]
print('EWMA win share against trailing 252d, name level:')
print(name_level[['method', 'scope', 'win_share', 'n_obs']].round(4).to_string(index=False))
by_year = window[(window.scope == 'day_level_by_year') & (window.method == 'ewma_097')]
print()
print('EWMA(0.97) by name-day, calendar year, full history from 2010:')
print(by_year[['year', 'win_share', 'n_obs']].round(4).to_string(index=False))


seeded sample: 100 names, seed 20260910
drawn from the names with full coverage over the window
fits that converged: 98 not converged: ['LW', 'RDDT']

 horizon      method  n_names  win_share  mean_qlike  baseline_qlike
       1    ewma_094      483     0.3540     -6.7009         -6.7487
       1    ewma_097      483     0.5197     -6.7436         -6.7487
       1       garch       98     0.6122     -6.6649         -6.6532
       1 trailing_63      611     0.3666     -6.5828         -6.6204
      21    ewma_094      483     0.1988     -3.5884         -3.6759
      21    ewma_097      483     0.3437     -3.6488         -3.6759
      21       garch       98     0.4898     -3.5585         -3.5752
      21 trailing_63      611     0.3077     -3.5134         -3.5479

GARCH reaches 61.2% at one step here and 49.0% at 21 days, against 46.7% and
55.6% on the alphabetical 60 F2.3b first used, so the win share depends on
which names are tested. EWMA(0.94) is far below 60% either way.

EWMA win s

In [17]:
rolling = pd.read_parquet(ROOT / 'data/eval/momentum_exposure_rolling.parquet')
stats = params['mom_exposure']
print('MOM exposure at each rebalance, from rolling 252d name-level')
print('betas dated at the rebalance and the weights actually held')
print()
print('rebalances:            ', stats['n_rebalances'])
print('mean exposure:         %+.4f' % stats['rolling_mean'])
print('range:                 %+.4f to %+.4f' % (stats['rolling_min'], stats['rolling_max']))
print('static full-sample:    %+.4f' % stats['static_aggregate'])
print('regression loading:    %+.4f' % stats['regression_loading'])
print('share from rolling betas, mean: %.4f' % stats['rolling_share_mean'])
print('share from full-sample betas:   %.4f' % stats['static_share_last_month'])
print()
print('the static aggregate says a momentum book has no momentum exposure; the')
print('conditional one moves with the book, which is what the book does.')
print()
risk = pd.read_parquet(ROOT / 'data/portfolios/seed_mom_ls_risk_21.parquet')
bias = risk['bias_ratio'].dropna()
by_year = bias.groupby(bias.index.year).mean()
print('diagonal model bias statistic for the momentum book, 21 days forward:')
print(by_year.round(4).to_string())
print()
print('mean %.4f, every year above 1.0, against 1.0225 for the equal-weight book.'
      % by_year.mean())


MOM exposure at each rebalance, from rolling 252d name-level
betas dated at the rebalance and the weights actually held

rebalances:             195
mean exposure:         +0.0698
range:                 -0.3322 to +0.3905
static full-sample:    -0.0162
regression loading:    +0.2885
share from rolling betas, mean: 0.7390
share from full-sample betas:   0.0968

the static aggregate says a momentum book has no momentum exposure; the
conditional one moves with the book, which is what the book does.

diagonal model bias statistic for the momentum book, 21 days forward:
date
2010    1.1261
2011    1.5782
2012    1.1063
2013    1.3015
2014    1.3647
2015    2.1180
2016    1.5325
2017    1.7224
2018    1.9529
2019    1.9597
2020    2.4560
2021    1.8116
2022    1.9603
2023    2.0911
2024    2.2668
2025    2.2504
2026    2.9259

mean 1.8544, every year above 1.0, against 1.0225 for the equal-weight book.
